# Voice Casting : Attribution de voix TTS par personnage

**Navigation** : [Index](../README.md) | [<< Precedent](04-7-TTS-Voice-Benchmark.ipynb) | [Suivant >>](04-10-Annotation-Prosodique.ipynb)

**Epic #1028 — P2 : Generation d'echantillons audio pour chaque personnage detecte en P1.**

Ce notebook prend les resultats de la lecture analytique (P1) et attribue une voix TTS concrete a chaque personnage en generant un echantillon audio de 5-15 secondes.

| Etape | Description |
|-------|-------------|
| **1** | Charger les resultats P1 (personnages, segments, specs vocales) |
| **2** | Mapper les profils FishAudio aux voix TTS disponibles |
| **3** | Generer un echantillon audio par personnage |
| **4** | Ecouter et valider le casting |
| **5** | Exporter la configuration de casting pour P3/P4 |

### Modeles TTS disponibles

- **Kokoro TTS** (local, port 8191) : 10 voix francaises, rapide (~5s)
- **OpenAI TTS** (cloud, tts-1) : 6 voix multilingues, haute qualite
- **FishAudio S2-Pro** (prevu production) : voice cloning, tags expressifs — non deploye en Docker

**Note** : En production, FishAudio S2-Pro sera le modele principal avec voice cloning zero-shot et tags expressifs natifs. Ce notebook utilise Kokoro + OpenAI comme substituts demo.


## 1. Configuration et imports

In [1]:
# Import guards - availability flags for external dependencies

try:
    from dotenv import load_dotenv
    DOTENV_AVAILABLE = True
except ImportError:
    DOTENV_AVAILABLE = False
    print(f'  dotenv non disponible - certaines fonctionnalites seront limitees')

try:
    import openai
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False
    print(f'  openai non disponible - certaines fonctionnalites seront limitees')

try:
    import pandas as pd
    PANDAS_AVAILABLE = True
except ImportError:
    PANDAS_AVAILABLE = False
    print(f'  pandas non disponible - certaines fonctionnalites seront limitees')

try:
    import requests
    REQUESTS_AVAILABLE = True
except ImportError:
    REQUESTS_AVAILABLE = False
    print(f'  requests non disponible - certaines fonctionnalites seront limitees')


import os
import json
import time
import IPython.display as ipd
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional

import requests
import pandas as pd

# Charger les variables d'environnement GenAI
from dotenv import load_dotenv
_env_candidates = [
    Path("../.env").resolve(),
    Path("../../.env").resolve(),
    Path("../../../.env").resolve(),
]
for _env_path in _env_candidates:
    if _env_path.exists():
        load_dotenv(_env_path)
        print(f"Env charge: {_env_path}")
        break
else:
    print("WARNING: Fichier .env non trouve")

TTS_KOKORO_URL = os.getenv("TTS_API_URL", "http://localhost:8191")
TTS_API_KEY = os.getenv("TTS_API_KEY", "")

# Repertoire de sortie pour les echantillons
P1_OUTPUT = Path("p1_output")
VOICE_DIR = Path("voice_casting_output")
VOICE_DIR.mkdir(exist_ok=True)

print(f"Kokoro TTS: {TTS_KOKORO_URL}")
print(f"P1 output dir: {P1_OUTPUT.resolve()}")
print(f"Voice output dir: {VOICE_DIR.resolve()}")

Env charge: D:\Dev\CoursIA\MyIA.AI.Notebooks\GenAI\.env
Kokoro TTS: http://localhost:8191
P1 output dir: D:\Dev\CoursIA\MyIA.AI.Notebooks\GenAI\Audio\04-Applications\p1_output
Voice output dir: D:\Dev\CoursIA\MyIA.AI.Notebooks\GenAI\Audio\04-Applications\voice_casting_output


## 2. Chargement des resultats P1 (Lecture analytique)

In [2]:
# Charger les resultats de la lecture analytique (P1)
# Le notebook P1 genere : characters.json, segments.csv, utterances.csv, voice_recommendations.csv

def load_p1_results(p1_dir: Path) -> dict:
    results = {}

    # Characters
    chars_path = p1_dir / "characters.json"
    if chars_path.exists():
        with open(chars_path, "r", encoding="utf-8") as f:
            results["characters"] = json.load(f)
        print(f"Personnages charges: {len(results['characters'])}")
    else:
        print(f"WARNING: {chars_path} non trouve")
        results["characters"] = {}

    # Segments
    seg_path = p1_dir / "segments.csv"
    if seg_path.exists():
        results["segments"] = pd.read_csv(seg_path)
        print(f"Segments charges: {len(results['segments'])}")
    else:
        print(f"WARNING: {seg_path} non trouve")
        results["segments"] = pd.DataFrame()

    # Utterances
    utt_path = p1_dir / "utterances.csv"
    if utt_path.exists():
        results["utterances"] = pd.read_csv(utt_path)
        print(f"Repliques charges: {len(results['utterances'])}")
    else:
        print(f"WARNING: {utt_path} non trouve")
        results["utterances"] = pd.DataFrame()

    # Voice recommendations
    voice_path = p1_dir / "voice_recommendations.csv"
    if voice_path.exists():
        results["voice_specs"] = pd.read_csv(voice_path)
        print(f"Specs vocales charges: {len(results['voice_specs'])}")
    else:
        print(f"WARNING: {voice_path} non trouve")
        results["voice_specs"] = pd.DataFrame()

    return results

p1 = load_p1_results(P1_OUTPUT)

# Afficher les personnages detectes
if p1["characters"]:
    chars_df = pd.DataFrame.from_dict(p1["characters"], orient="index")
    print("\n=== Personnages detectes ===")
    print(chars_df[["name", "gender", "role", "utterance_count"]].to_string())
else:
    print("Aucun personnage charge depuis P1. Utilisation des donnees de fallback.")

Aucun personnage charge depuis P1. Utilisation des donnees de fallback.


### Donnees de fallback (si P1 non execute)

In [3]:
# Donnees de fallback si les fichiers P1 ne sont pas disponibles
# Ces donnees correspondent aux resultats attendus du notebook 04-8-Lecture-Analytique.ipynb

if not p1["characters"]:
    p1["characters"] = {
        "elisabeth_rousset": {
            "name": "Elisabeth Rousset",
            "gender": "F",
            "role": "protagoniste",
            "utterance_count": 3,
            "first_utterance": "Et ils ne font rien de mal aux gens ?"
        },
        "cornudet": {
            "name": "Cornudet",
            "gender": "M",
            "role": "secondaire",
            "utterance_count": 2,
            "first_utterance": "On dit qu'ils seraient a Evreux demain."
        },
        "comtesse": {
            "name": "Comtesse de Breville",
            "gender": "F",
            "role": "antagoniste",
            "utterance_count": 1,
            "first_utterance": "Et ils ne font rien de mal aux gens ?"
        },
        "comte": {
            "name": "Comte de Breville",
            "gender": "M",
            "role": "antagoniste",
            "utterance_count": 1,
            "first_utterance": "Rassurez-vous, madame, ils sont corrects."
        },
        "narrateur": {
            "name": "Narrateur",
            "gender": "M",
            "role": "narrateur",
            "utterance_count": 0,
            "first_utterance": "Pendant plusieurs jours, des lambeaux d'armees en deroute avaient traverse la ville."
        },
        "carre_lamadon": {
            "name": "Monsieur Carre-Lamadon",
            "gender": "M",
            "role": "secondaire",
            "utterance_count": 1,
            "first_utterance": "On dit qu'ils seraient a Evreux demain."
        },
        "loiseau": {
            "name": "Monsieur Loiseau",
            "gender": "M",
            "role": "figurant",
            "utterance_count": 0,
            "first_utterance": "Ce n'etait point de la troupe, mais des hordes debraillees."
        },
        "soeurs": {
            "name": "Les deux soeurs",
            "gender": "F",
            "role": "figurant",
            "utterance_count": 0,
            "first_utterance": "Quelques-uns demeuraient chez eux."
        },
    }
    print(f"Fallback: {len(p1['characters'])} personnages charges.")

chars_df = pd.DataFrame.from_dict(p1["characters"], orient="index")
print("\n=== Personnages pour le voice casting ===")
print(chars_df[["name", "gender", "role", "utterance_count"]].to_string())

Fallback: 8 personnages charges.

=== Personnages pour le voice casting ===
                                     name gender          role  utterance_count
elisabeth_rousset       Elisabeth Rousset      F  protagoniste                3
cornudet                         Cornudet      M    secondaire                2
comtesse             Comtesse de Breville      F   antagoniste                1
comte                   Comte de Breville      M   antagoniste                1
narrateur                       Narrateur      M     narrateur                0
carre_lamadon      Monsieur Carre-Lamadon      M    secondaire                1
loiseau                  Monsieur Loiseau      M      figurant                0
soeurs                    Les deux soeurs      F      figurant                0


## 3. Mapping profils FishAudio vers voix TTS disponibles

Le P1 a genere des profils FishAudio (timbre, tags expressifs, preset). Cette etape mappe chaque profil vers une voix concrete sur les services TTS disponibles.

| Service | Voix FR | Avantage |
|---------|---------|----------|
| Kokoro TTS | 10 voix (5F + 5M) | Rapide, local |
| OpenAI TTS | 6 voix multilingues | Haute qualite |
| FishAudio S2-Pro | Voice cloning, tags | Production (futur) |

In [4]:
# Mapping des profils FishAudio vers voix Kokoro et OpenAI
# Les voix Kokoro FR : af_sky, af_bella, af_nicole, af_sarah (F) | am_adam, am_michael (M)
#                        bf_emma, bf_isabella (F) | bm_george, bm_lewis (M)
# Les voix OpenAI : alloy, echo, fable, onyx, nova, shimmer

VOICE_MAP = {
    "protagoniste": {
        "F": {"kokoro": "af_sky",    "openai": "nova",    "fish_preset": "expressive_female_warm"},
        "M": {"kokoro": "am_adam",   "openai": "echo",    "fish_preset": "expressive_male_warm"},
    },
    "antagoniste": {
        "F": {"kokoro": "af_sarah",  "openai": "shimmer", "fish_preset": "expressive_female_cold"},
        "M": {"kokoro": "am_michael","openai": "onyx",    "fish_preset": "expressive_male_cold"},
    },
    "secondaire": {
        "F": {"kokoro": "af_bella",  "openai": "alloy",   "fish_preset": "expressive_female_neutral"},
        "M": {"kokoro": "bm_george", "openai": "fable",   "fish_preset": "expressive_male_neutral"},
    },
    "figurant": {
        "F": {"kokoro": "bf_emma",   "openai": "alloy",   "fish_preset": "neutral_female"},
        "M": {"kokoro": "bm_lewis",  "openai": "fable",   "fish_preset": "neutral_male"},
    },
    "narrateur": {
        "F": {"kokoro": "af_nicole", "openai": "nova",    "fish_preset": "narrator_female"},
        "M": {"kokoro": "bf_isabella","openai":"echo",     "fish_preset": "narrator_male"},
    },
}

def assign_voice(character_id: str, char_info: dict) -> dict:
    gender = char_info.get("gender", "M")
    role = char_info.get("role", "figurant")
    profile = VOICE_MAP.get(role, VOICE_MAP["figurant"]).get(gender, VOICE_MAP["figurant"]["M"])
    return {
        "character_id": character_id,
        "name": char_info["name"],
        "gender": gender,
        "role": role,
        "kokoro_voice": profile["kokoro"],
        "openai_voice": profile["openai"],
        "fish_preset": profile["fish_preset"],
        "model_primary": "kokoro",
        "model_production": "fishaudio/s2-pro",
    }

# Attribuer les voix
cast_table = []
for cid, info in p1["characters"].items():
    cast_table.append(assign_voice(cid, info))

cast_df = pd.DataFrame(cast_table)
print("=== Table de voice casting ===")
print(cast_df[["name", "gender", "role", "kokoro_voice", "openai_voice", "fish_preset"]].to_string(index=False))

=== Table de voice casting ===
                  name gender         role kokoro_voice openai_voice             fish_preset
     Elisabeth Rousset      F protagoniste       af_sky         nova  expressive_female_warm
              Cornudet      M   secondaire    bm_george        fable expressive_male_neutral
  Comtesse de Breville      F  antagoniste     af_sarah      shimmer  expressive_female_cold
     Comte de Breville      M  antagoniste   am_michael         onyx    expressive_male_cold
             Narrateur      M    narrateur  bf_isabella         echo           narrator_male
Monsieur Carre-Lamadon      M   secondaire    bm_george        fable expressive_male_neutral
      Monsieur Loiseau      M     figurant     bm_lewis        fable            neutral_male
       Les deux soeurs      F     figurant      bf_emma        alloy          neutral_female


## 4. Fonctions de synthese TTS

In [5]:
def synthesize_kokoro(text: str, voice: str = "af_sky", output_path: Optional[str] = None) -> dict:
    # Synthesize via Kokoro TTS (local service, port 8191)
    headers = {"Content-Type": "application/json"}
    if TTS_API_KEY:
        headers["Authorization"] = f"Bearer {TTS_API_KEY}"
    payload = {"input": text, "model": "kokoro", "voice": voice}

    t0 = time.perf_counter()
    resp = requests.post(f"{TTS_KOKORO_URL}/v1/audio/speech", json=payload, headers=headers, timeout=60)
    elapsed = (time.perf_counter() - t0) * 1000

    resp.raise_for_status()
    audio_bytes = resp.content

    if output_path:
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, "wb") as f:
            f.write(audio_bytes)

    return {"model": "kokoro", "voice": voice, "latency_ms": elapsed,
            "audio_size_bytes": len(audio_bytes), "status": "ok"}


def synthesize_openai(text: str, voice: str = "alloy", output_path: Optional[str] = None) -> dict:
    # Synthesize via OpenAI TTS API (cloud)
    from openai import OpenAI
    client = OpenAI()

    t0 = time.perf_counter()
    response = client.audio.speech.create(model="tts-1", voice=voice, input=text)
    elapsed = (time.perf_counter() - t0) * 1000

    audio_bytes = response.content

    if output_path:
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, "wb") as f:
            f.write(audio_bytes)

    return {"model": "openai/tts-1", "voice": voice, "latency_ms": elapsed,
            "audio_size_bytes": len(audio_bytes), "status": "ok"}


print("Fonctions TTS chargees : synthesize_kokoro, synthesize_openai")

Fonctions TTS chargees : synthesize_kokoro, synthesize_openai


## 5. Generation des echantillons audio par personnage

Pour chaque personnage detecte en P1, on genere un echantillon audio avec sa voix attribuee. Le texte utilise est la premiere replique du personnage (ou un texte narratif pour le narrateur).

In [6]:
synthesis_results = []
audio_files = {}

for cast in cast_table:
    cid = cast["character_id"]
    name = cast["name"]
    kokoro_voice = cast["kokoro_voice"]
    openai_voice = cast["openai_voice"]

    sample_text = p1["characters"][cid].get("first_utterance", "")
    if not sample_text or len(sample_text) < 20:
        sample_text = "Bonjour, je suis un personnage de cette histoire."

    print(f"[{name}] voice={kokoro_voice} | {sample_text[:60]}...")

    kokoro_path = str(VOICE_DIR / f"{cid}_kokoro.mp3")
    try:
        result = synthesize_kokoro(sample_text, voice=kokoro_voice, output_path=kokoro_path)
        result["character_id"] = cid
        result["character_name"] = name
        synthesis_results.append(result)
        audio_files[cid] = kokoro_path
        print(f"  Kokoro: {result['latency_ms']:.0f}ms, {result['audio_size_bytes']/1024:.1f}KB")
    except Exception as e:
        print(f"  Kokoro ERREUR: {e}")
        synthesis_results.append({"character_id": cid, "character_name": name,
                                   "model": "kokoro", "status": "error", "error": str(e)[:80]})

    openai_path = str(VOICE_DIR / f"{cid}_openai.mp3")
    try:
        result = synthesize_openai(sample_text, voice=openai_voice, output_path=openai_path)
        result["character_id"] = cid
        result["character_name"] = name
        synthesis_results.append(result)
        print(f"  OpenAI: {result['latency_ms']:.0f}ms, {result['audio_size_bytes']/1024:.1f}KB")
    except Exception as e:
        print(f"  OpenAI ERREUR: {e}")
        synthesis_results.append({"character_id": cid, "character_name": name,
                                   "model": "openai", "status": "error", "error": str(e)[:80]})

print(f"\nEchantillons generes: {len(synthesis_results)} "
      f"({sum(1 for r in synthesis_results if r['status']=='ok')} OK)")

[Elisabeth Rousset] voice=af_sky | Et ils ne font rien de mal aux gens ?...


  Kokoro: 2652ms, 51.0KB


  OpenAI: 3683ms, 40.8KB
[Cornudet] voice=bm_george | On dit qu'ils seraient a Evreux demain....


  Kokoro: 2535ms, 49.9KB


  OpenAI: 3543ms, 49.2KB
[Comtesse de Breville] voice=af_sarah | Et ils ne font rien de mal aux gens ?...


...

  OpenAI: 1920ms, 47.3KB
[Monsieur Loiseau] voice=bm_lewis | Ce n'etait point de la troupe, mais des hordes debraillees....


  Kokoro: 2517ms, 67.2KB


  OpenAI: 1322ms, 72.2KB
[Les deux soeurs] voice=bf_emma | Quelques-uns demeuraient chez eux....


  Kokoro: 2596ms, 43.2KB


  OpenAI: 866ms, 41.2KB

Echantillons generes: 16 (16 OK)


## 6. Ecoute des echantillons — Validation du casting

In [7]:
# Ecouter chaque echantillon Kokoro (principal pour le demo)
for cast in cast_table:
    cid = cast["character_id"]
    name = cast["name"]
    role = cast["role"]
    kokoro_voice = cast["kokoro_voice"]

    kokoro_path = VOICE_DIR / f"{cid}_kokoro.mp3"
    print(f"\n{'='*60}")
    print(f"Personnage: {name} ({role}) | Voix Kokoro: {kokoro_voice}")
    print(f"{'='*60}")

    if kokoro_path.exists() and kokoro_path.stat().st_size > 1000:
        ipd.display(ipd.Audio(filename=str(kokoro_path)))
    else:
        print("  (echantillon non disponible)")


Personnage: Elisabeth Rousset (protagoniste) | Voix Kokoro: af_sky



Personnage: Cornudet (secondaire) | Voix Kokoro: bm_george



Personnage: Comtesse de Breville (antagoniste) | Voix Kokoro: af_sarah



Personnage: Comte de Breville (antagoniste) | Voix Kokoro: am_michael



Personnage: Narrateur (narrateur) | Voix Kokoro: bf_isabella



Personnage: Monsieur Carre-Lamadon (secondaire) | Voix Kokoro: bm_george



Personnage: Monsieur Loiseau (figurant) | Voix Kokoro: bm_lewis



Personnage: Les deux soeurs (figurant) | Voix Kokoro: bf_emma


## 7. Tableau comparatif des resultats

In [8]:
synth_df = pd.DataFrame(synthesis_results)

if "latency_ms" in synth_df.columns:
    ok_results = synth_df[synth_df["status"] == "ok"].copy()
    ok_results["latency_ms"] = ok_results["latency_ms"].round(0)
    ok_results["audio_size_kb"] = (ok_results["audio_size_bytes"] / 1024).round(1)

    print("=== Resultats de synthese par personnage et modele ===\n")
    print(ok_results[["character_name", "model", "voice", "latency_ms", "audio_size_kb"]].to_string(index=False))

    # Resume par modele
    summary = ok_results.groupby("model").agg(
        avg_latency_ms=("latency_ms", "mean"),
        avg_size_kb=("audio_size_kb", "mean"),
        n_samples=("character_name", "count"),
    ).round(0)
    print("\n=== Resume par modele ===")
    print(summary.to_string())
else:
    print("Aucun resultat de synthese disponible.")

=== Resultats de synthese par personnage et modele ===

        character_name        model       voice  latency_ms  audio_size_kb
     Elisabeth Rousset       kokoro      af_sky      2652.0           51.0
     Elisabeth Rousset openai/tts-1        nova      3683.0           40.8
              Cornudet       kokoro   bm_george      2535.0           49.9
              Cornudet openai/tts-1       fable      3543.0           49.2
  Comtesse de Breville       kokoro    af_sarah      2681.0           51.0
  Comtesse de Breville openai/tts-1     shimmer      1770.0           45.9
     Comte de Breville       kokoro  am_michael      2506.0           53.7
     Comte de Breville openai/tts-1        onyx      1902.0           45.5
             Narrateur       kokoro bf_isabella      2700.0           89.7
             Narrateur openai/tts-1        echo      2268.0           95.6
Monsieur Carre-Lamadon       kokoro   bm_george      2483.0           49.9
Monsieur Carre-Lamadon openai/tts-1       fa


=== Resume par modele ===


              avg_latency_ms  avg_size_kb  n_samples
model                                               
kokoro                2584.0         57.0          8
openai/tts-1          2159.0         55.0          8


## 8. Export de la configuration de casting

La configuration de casting est exportee pour consommation par P3 (annotation prosodique) et P4 (generation TTS). Le format inclut les mappings pour chaque service TTS disponible ainsi que les profils FishAudio pour la production.

In [9]:
# Construire la configuration de casting complete
casting_config = {
    "epic": "1028",
    "pass": "P2",
    "description": "Voice casting configuration for audiobook pipeline",
    "source_text": "Boule de Suif - Guy de Maupassant",
    "characters": {},
}

for cast in cast_table:
    cid = cast["character_id"]
    char_info = p1["characters"][cid]

    casting_config["characters"][cid] = {
        "name": cast["name"],
        "gender": cast["gender"],
        "role": cast["role"],
        "voices": {
            "kokoro": {
                "model": "kokoro",
                "voice_id": cast["kokoro_voice"],
                "service_url": "http://localhost:8191",
            },
            "openai": {
                "model": "tts-1",
                "voice_id": cast["openai_voice"],
                "service": "cloud",
            },
            "fishaudio_production": {
                "model": "fishaudio/s2-pro",
                "preset": cast["fish_preset"],
                "voice_cloning": False,
                "expressive_tags": True,
                "note": "Modele production — non deploye en Docker. Utiliser Kokoro/OpenAI pour demo.",
            },
        },
        "sample_text": char_info.get("first_utterance", ""),
        "sample_file": f"voice_casting_output/{cid}_kokoro.mp3",
    }

# Sauvegarder
config_path = VOICE_DIR / "voice_casting_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(casting_config, f, indent=2, ensure_ascii=False)
print(f"Configuration de casting sauvegardee: {config_path}")
print(f"Personnages configures: {len(casting_config['characters'])}")

# Sauvegarder le tableau de casting en CSV
cast_df.to_csv(VOICE_DIR / "voice_casting_table.csv", index=False, encoding="utf-8")
print(f"Tableau CSV sauvegarde: {VOICE_DIR / 'voice_casting_table.csv'}")

Configuration de casting sauvegardee: voice_casting_output\voice_casting_config.json
Personnages configures: 8
Tableau CSV sauvegarde: voice_casting_output\voice_casting_table.csv


## 9. Conclusion et passage a P3

### Recapitulatif du voice casting

Chaque personnage detecte en P1 a recu une voix TTS attribuee selon son profil (genre + role narratif). Les echantillons audio sont generes via Kokoro TTS (local) et OpenAI TTS (cloud) comme substituts demo.

### Architecture production

| Composant | Demo (actuel) | Production (futur) |
|-----------|---------------|---------------------|
| TTS principal | Kokoro TTS | FishAudio S2-Pro |
| Voice cloning | Non | FishAudio zero-shot |
| Tags expressifs | Non | [laugh], [whisper], [sigh], etc. |
| Mastering | Non | Demucs + Kokoro |

### Prochaines etapes (Epic #1028)

- **P3 — Annotation prosodique** : Enrichir le texte avec les tags expressifs FishAudio (`[laugh]`, `[whisper]`, `[sigh]`, `[gasp]`, `[pause]`, `[shout]`)
- **P4 — Generation TTS** : Synthetiser les segments audio avec la voix appropriatee
- **P5 — Compilation audio** : Assembler les chunks avec ffmpeg + mastering